In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = '/mnt/data/DarkNet.csv'  # Ενημερωμένη διαδρομή του αρχείου

data = pd.read_csv(file_path)

# Preprocessing: Cleaning 'Flow_Bytes/s'
# Σκοπός: Η στήλη 'Flow_Bytes/s' περιέχει τιμές που μπορεί να μην είναι αριθμητικές, γεγονός που μπορεί να προκαλέσει σφάλματα στην ανάλυση. 
# Μετατρέπουμε αυτές τις τιμές σε αριθμητικές και αφαιρούμε τυχόν μη έγκυρες εγγραφές.
data['Flow_Bytes/s'] = pd.to_numeric(data['Flow_Bytes/s'], errors='coerce')
data.dropna(subset=['Flow_Bytes/s'], inplace=True)

# Extracting features and labels
# Βεβαιωθείτε ότι οι στήλες ['Label-1', 'Label-2'] είναι οι σωστές ετικέτες για αφαίρεση. Ελέγξτε την ακριβή ονομασία τους στο dataset.
X = data.drop(columns=['Label-1', 'Label-2'])  # Drop target columns
# Αντιστοιχίστε τις ετικέτες Tor και Non-Tor σε δυαδικές τιμές για την ταξινόμηση
y = data['Label-1'].map({'Tor': 1, 'Non-Tor': 0})  # Encode labels as 1 (Tor) and 0 (Non-Tor)

# Check for categorical features
# Εξετάστε αν υπάρχουν κατηγορικές στήλες που χρειάζονται κωδικοποίηση πριν την κανονικοποίηση.
categorical_columns = X.select_dtypes(include=['object', 'category']).columns
if not categorical_columns.empty:
    print("Προσοχή: Υπάρχουν κατηγορικές στήλες που χρειάζονται κωδικοποίηση:", categorical_columns.tolist())

# Normalize numerical features
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X.select_dtypes(include=['float64', 'int64']))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

# Model training: Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predictions and Evaluation
y_pred = rf_model.predict(X_test)

# Classification Report
print("Classification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm / cm.sum(axis=1)[:, None], annot=True, fmt='.2%', cmap='Blues', xticklabels=['Non-Tor', 'Tor'], yticklabels=['Non-Tor', 'Tor'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (with Percentages)')
plt.show()
